# Generate LLM Embeddings using GPT-4.0-mini via OpenAI API with .env support

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [7]:
# ... (load client and data as before)
df = pd.read_csv("../data/processed/train_full.csv").head(1000)

In [8]:
# Create an analytical prompt
def create_analytical_prompt(row):
    prompt = f"Analyze the following transaction: Product {row.get('ProductCD', 'NA')}, "
    prompt += f"card {row.get('card1', 'NA')}, address {row.get('addr1', 'NA')}, "
    prompt += f"device {row.get('DeviceType', 'NA')} ({row.get('DeviceInfo', 'NA')}), "
    prompt += f"sender's email {row.get('P_emaildomain', 'NA')}. "
    prompt += "What are its key characteristics?"
    return prompt

In [9]:
# Generate embeddings (not text)
results = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = create_analytical_prompt(row)
    try:
        # FIXED: Use the embeddings API, not the chat completions API
        response = client.embeddings.create(
            model="text-embedding-3-small",  # This is a good, low-cost embedding model
            input=prompt
        )
        embedding = response.data[0].embedding
    except Exception as e:
        embedding = [0.0] * 1536 # Default to zeros on error
    
    results.append({
        "TransactionID": row['TransactionID'],
        "Prompt": prompt,
        "LLM_Embedding": embedding
    })

100%|██████████| 1000/1000 [07:32<00:00,  2.21it/s]


In [3]:
 # Save the embeddings
output_df = pd.DataFrame(results)
# Flatten the list of embeddings into a single dataframe
output_df_flat = pd.json_normalize(output_df['LLM_Embedding']).add_prefix('LLM_embed_')
output_df = pd.concat([output_df.drop('LLM_Embedding', axis=1), output_df_flat], axis=1)

output_df.to_csv("../data/processed/llm_embeddings.csv", index=False)

NameError: name 'results' is not defined